In [1]:
import threading
import time
import random

# --- 1. Initialize our Synchronization Flags ---
data_available_event = threading.Event()  # Signals when a new data frame is ready (True/Green)
stop_event = threading.Event()            # Signals when the entire system needs to shut down

# Shared Memory Bridge (Simulating target coordinates for the robot arm)
shared_sensor_data = None

def sensor_stream_worker():
    """
    PRODUCER THREAD: Simulates a sensor array or VLM computing engine
    generating fresh spatial coordinates at uneven intervals.
    """
    global shared_sensor_data
    print(f"[{threading.current_thread().name}] Sensor streaming thread initialized.")

    frame_counter = 1
    while not stop_event.is_set():
        # Simulate uneven time gaps between data packets arriving (e.g., 0.4 to 0.9 seconds)
        time.sleep(random.uniform(0.4, 0.9))

        if stop_event.is_set():
            break

        # Simulate data generation (e.g., a mock random X, Y, Z coordinate packet)
        shared_sensor_data = {
            "frame_id": frame_counter,
            "coordinates": (round(random.uniform(-10, 10), 2), round(random.uniform(-10, 10), 2))
        }
        print(f"\n[{threading.current_thread().name}] Generated Data Frame #{frame_counter}: {shared_sensor_data['coordinates']}")

        # Trigger signal: Wake up the processing thread
        data_available_event.set()

        # Instantly clear it so it acts like a brief pulse notification for this frame
        data_available_event.clear()

        frame_counter += 1

    print(f"[{threading.current_thread().name}] Streaming thread shutting down cleanly.")


def data_processor_worker():
    """
    CONSUMER THREAD: Simulates the IK loop thread waiting efficiently
    until fresh spatial targets land, executing motor translations.
    """
    print(f"[{threading.current_thread().name}] Processing thread initialized. Idle state active (0% CPU).")

    while not stop_event.is_set():
        # Block here efficiently until data_available_event.set() is called
        # We pass a brief timeout so it checks the stop_event loop state periodically
        signaled = data_available_event.wait(timeout=0.1)

        if signaled and not stop_event.is_set():
            # Critical Section execution: Read data out of shared memory bridge safely
            current_frame = shared_sensor_data["frame_id"]
            current_coords = shared_sensor_data["coordinates"]

            print(f"[{threading.current_thread().name}] Event Caught! Processing Frame #{current_frame}. Moving motors to: {current_coords}")
            # Simulate processing calculation / mechanical manipulation delay
            time.sleep(0.1)

    print(f"[{threading.current_thread().name}] Processor loop broken. Hardware parked safely.")

# --- Execution Simulation ---
print("[Main] Starting complete robotic subsystem middleware simulation...")

# Define and pair threads
sensor_thread = threading.Thread(target=sensor_stream_worker, name="Sensor_VLM_Stream")
processor_thread = threading.Thread(target=data_processor_worker, name="IK_Motor_Loop")

# Spin them up
sensor_thread.start()
processor_thread.start()

# Let them trade data streams for 4 seconds live
time.sleep(4.0)

# Trigger a clean absolute system shutdown
print("\n[Main] Simulation duration reached. Throwing main global stop_event flag...")
stop_event.set()

# Lock up the main program exit until background workers park safely
sensor_thread.join()
processor_thread.join()

print("[Main] Simulation closed down with 100% data integrity verified.")

[Main] Starting complete robotic subsystem middleware simulation...
[Sensor_VLM_Stream] Sensor streaming thread initialized.
[IK_Motor_Loop] Processing thread initialized. Idle state active (0% CPU).

[Sensor_VLM_Stream] Generated Data Frame #1: (8.74, 7.29)

[Sensor_VLM_Stream] Generated Data Frame #2: (-0.31, -3.04)
[IK_Motor_Loop] Event Caught! Processing Frame #2. Moving motors to: (-0.31, -3.04)

[Sensor_VLM_Stream] Generated Data Frame #3: (-0.0, -0.73)
[IK_Motor_Loop] Event Caught! Processing Frame #3. Moving motors to: (-0.0, -0.73)

[Sensor_VLM_Stream] Generated Data Frame #4: (1.93, -7.79)
[IK_Motor_Loop] Event Caught! Processing Frame #4. Moving motors to: (1.93, -7.79)

[Sensor_VLM_Stream] Generated Data Frame #5: (5.94, 7.96)
[IK_Motor_Loop] Event Caught! Processing Frame #5. Moving motors to: (5.94, 7.96)

[Main] Simulation duration reached. Throwing main global stop_event flag...
[IK_Motor_Loop] Processor loop broken. Hardware parked safely.
[Sensor_VLM_Stream] Streaming